# Análisis Exploratorio de Datos (EDA)
## Airbnb Ciudad Autónoma de Buenos Aires, Argentina
### Inteligencia de Negocios — Taller Evaluativo 2

**Objetivo:** Comprender la estructura, calidad y distribución de los datos antes de realizar las transformaciones del proceso ETL.

---

In [8]:
import sys
import os
sys.path.append(os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from extraccion import Extraccion

# Configuracion general de graficas
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


---
## 1. Extracción de datos desde MongoDB

In [10]:
# Conectar a MongoDB y extraer las tres colecciones
ext = Extraccion()
df_listings, df_reviews, df_calendar = ext.extraer_todo()
ext.cerrar_conexion()

print(f'Listings:  {len(df_listings):,} registros')
print(f'Reviews:   {len(df_reviews):,} registros')
print(f'Calendar:  {len(df_calendar):,} registros')

2026-04-04 14:43:16 - INFO - Logger iniciado para el modulo: extraccion
2026-04-04 14:43:16 - INFO - Archivo de log: logs\log_extraccion_20260404_1443.txt
2026-04-04 14:43:16 - INFO - Intentando conectar a MongoDB: mongodb://localhost:27017/
2026-04-04 14:43:18 - INFO - Conexion exitosa a la base de datos: 'airbnb_buenosaires'
2026-04-04 14:43:18 - INFO - Iniciando extraccion completa de todas las colecciones.
2026-04-04 14:43:18 - INFO - Extrayendo coleccion: 'listings'
2026-04-04 14:43:18 - WARNING - La coleccion 'listings' esta vacia o no existe.
2026-04-04 14:43:18 - INFO - Extrayendo coleccion: 'reviews'
2026-04-04 14:43:18 - WARNING - La coleccion 'reviews' esta vacia o no existe.
2026-04-04 14:43:18 - INFO - Extrayendo coleccion: 'calendar'
2026-04-04 14:43:18 - WARNING - La coleccion 'calendar' esta vacia o no existe.
2026-04-04 14:43:18 - INFO - Extraccion finalizada. Resumen: Listings=0 | Reviews=0 | Calendar=0
2026-04-04 14:43:18 - INFO - Conexion a MongoDB cerrada correctam

Listings:  0 registros
Reviews:   0 registros
Calendar:  0 registros


---
## 2. Entendimiento general — Listings

El dataset de **listings** contiene información detallada de cada alojamiento publicado en Airbnb para Buenos Aires.

In [ ]:
# Primeras filas
df_listings.head()

In [ ]:
# Dimensiones
print(f'Shape: {df_listings.shape}')
print(f'Registros: {df_listings.shape[0]:,}')
print(f'Columnas: {df_listings.shape[1]}')

In [ ]:
# Tipos de datos
df_listings.info()

In [ ]:
# Estadísticas descriptivas de variables numéricas clave
cols_numericas = ['price', 'minimum_nights', 'maximum_nights',
                  'availability_365', 'number_of_reviews', 'review_scores_rating']
cols_presentes = [c for c in cols_numericas if c in df_listings.columns]
df_listings[cols_presentes].describe()

---
## 3. Entendimiento general — Reviews

In [ ]:
df_reviews.head()

In [ ]:
print(f'Shape: {df_reviews.shape}')
df_reviews.info()

---
## 4. Entendimiento general — Calendar

In [ ]:
df_calendar.head()

In [ ]:
print(f'Shape: {df_calendar.shape}')
df_calendar.info()

---
## 5. Calidad de datos — Valores nulos

In [ ]:
def resumen_nulos(df, nombre):
    """Muestra el resumen de valores nulos por columna con porcentaje."""
    nulos = df.isnull().sum()
    pct   = (nulos / len(df) * 100).round(2)
    resumen = pd.DataFrame({'Nulos': nulos, '% Nulos': pct})
    resumen = resumen[resumen['Nulos'] > 0].sort_values('% Nulos', ascending=False)
    print(f'\n--- Valores nulos: {nombre} ---')
    print(resumen.to_string())
    return resumen

nulos_listings  = resumen_nulos(df_listings, 'Listings')
nulos_reviews   = resumen_nulos(df_reviews,  'Reviews')
nulos_calendar  = resumen_nulos(df_calendar, 'Calendar')

In [ ]:
# Visualización de nulos en Listings (top 20 columnas con más nulos)
if not nulos_listings.empty:
    top20 = nulos_listings.head(20)
    fig, ax = plt.subplots(figsize=(12, 6))
    top20['% Nulos'].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 20 columnas con más valores nulos — Listings', fontsize=13)
    ax.set_xlabel('% de valores nulos')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print('Interpretación: Las columnas con alto % de nulos como neighbourhood o review_scores\n'
          'deberán ser tratadas o excluidas en la fase de transformación.')

---
## 6. Calidad de datos — Duplicados

In [ ]:
for nombre, df in [('Listings', df_listings), ('Reviews', df_reviews), ('Calendar', df_calendar)]:
    dup = df.duplicated().sum()
    print(f'{nombre}: {dup:,} registros duplicados ({dup/len(df)*100:.2f}%)')

print('\nDecisión: Se eliminarán los duplicados exactos en la fase de transformación,'
      ' ya que no aportan información adicional y pueden sesgar los análisis.')

---
## 7. Análisis de outliers — Price, minimum_nights, availability_365

In [ ]:
# Normalizar price para el análisis exploratorio
df_listings['price_num'] = (
    df_listings['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_listings['price_num'] = pd.to_numeric(df_listings['price_num'], errors='coerce')
price_valido = df_listings['price_num'].dropna()

print('Estadísticas de price (sin nulos):')
print(price_valido.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de price (sin outliers extremos)
precio_filtrado = price_valido[price_valido <= price_valido.quantile(0.95)]
axes[0].hist(precio_filtrado, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Price (percentil 95)', fontsize=12)
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')

# Boxplot de price
axes[1].boxplot(price_valido, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot de Price (con outliers)', fontsize=12)
axes[1].set_xlabel('Precio (USD)')

plt.tight_layout()
plt.show()
print('Interpretación: El campo price tiene una distribución muy sesgada hacia la derecha.\n'
      'Existen outliers con precios extremadamente altos que deberán analizarse.')

In [ ]:
# Análisis de minimum_nights y availability_365
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'minimum_nights' in df_listings.columns:
    mn = pd.to_numeric(df_listings['minimum_nights'], errors='coerce').dropna()
    mn_filtrado = mn[mn <= mn.quantile(0.95)]
    axes[0].hist(mn_filtrado, bins=40, color='coral', edgecolor='white')
    axes[0].set_title('Distribución de minimum_nights', fontsize=12)
    axes[0].set_xlabel('Mínimo de noches')
    axes[0].set_ylabel('Frecuencia')

if 'availability_365' in df_listings.columns:
    av = pd.to_numeric(df_listings['availability_365'], errors='coerce').dropna()
    axes[1].hist(av, bins=40, color='mediumseagreen', edgecolor='white')
    axes[1].set_title('Distribución de availability_365', fontsize=12)
    axes[1].set_xlabel('Días disponibles en el año')
    axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()
print('Interpretación: minimum_nights presenta outliers (valores como 365, 730 días mínimos)\n'
      'que podrían indicar alquileres de larga temporada o errores de carga.')

---
## 8. Distribución por tipo de habitación y barrio

In [ ]:
if 'room_type' in df_listings.columns:
    conteo = df_listings['room_type'].value_counts()
    fig, ax = plt.subplots(figsize=(8, 5))
    conteo.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Distribución por tipo de habitación', fontsize=12)
    ax.set_xlabel('Tipo')
    ax.set_ylabel('Cantidad')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
    print('Interpretación: El tipo de alojamiento más frecuente revela el perfil del mercado de Airbnb en Buenos Aires.')

In [ ]:
if 'neighbourhood_cleansed' in df_listings.columns:
    top_barrios = df_listings['neighbourhood_cleansed'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 5))
    top_barrios.plot(kind='barh', ax=ax, color='coral')
    ax.set_title('Top 15 barrios con más alojamientos', fontsize=12)
    ax.set_xlabel('Cantidad de alojamientos')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print('Interpretación: Palermo y Recoleta concentran la mayor oferta de alojamientos en Buenos Aires.')

---
## 9. Análisis de Reviews — Distribución temporal

In [ ]:
if 'date' in df_reviews.columns:
    df_reviews['date_dt'] = pd.to_datetime(df_reviews['date'], errors='coerce')
    df_reviews['anio'] = df_reviews['date_dt'].dt.year
    conteo_anio = df_reviews['anio'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    conteo_anio.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title('Cantidad de reviews por año', fontsize=12)
    ax.set_xlabel('Año')
    ax.set_ylabel('Cantidad de reviews')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    print('Interpretación: El volumen de reviews muestra la evolución de la actividad de Airbnb en Buenos Aires.')

---
## 10. Análisis de Calendar — Disponibilidad y precios por mes

In [ ]:
if 'date' in df_calendar.columns and 'price' in df_calendar.columns:
    df_cal = df_calendar.copy()
    df_cal['date_dt'] = pd.to_datetime(df_cal['date'], errors='coerce')
    df_cal['mes'] = df_cal['date_dt'].dt.month
    df_cal['price_num'] = (
        df_cal['price'].astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_cal['price_num'] = pd.to_numeric(df_cal['price_num'], errors='coerce')

    precio_por_mes = df_cal.groupby('mes')['price_num'].median()

    fig, ax = plt.subplots(figsize=(12, 5))
    precio_por_mes.plot(kind='line', marker='o', ax=ax, color='steelblue', linewidth=2)
    ax.set_title('Precio mediano por mes — Calendar', fontsize=12)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Precio mediano (USD)')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
    plt.tight_layout()
    plt.show()
    print('Interpretación: La estacionalidad de precios permite identificar temporadas altas y bajas en Buenos Aires.')

In [ ]:
# Disponibilidad por mes
if 'available' in df_calendar.columns:
    df_cal2 = df_calendar.copy()
    df_cal2['date_dt'] = pd.to_datetime(df_cal2['date'], errors='coerce')
    df_cal2['mes'] = df_cal2['date_dt'].dt.month
    df_cal2['disponible'] = df_cal2['available'].map({'t': 1, 'f': 0})
    disp_mes = df_cal2.groupby('mes')['disponible'].mean() * 100

    fig, ax = plt.subplots(figsize=(12, 5))
    disp_mes.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
    ax.set_title('% de disponibilidad por mes — Calendar', fontsize=12)
    ax.set_xlabel('Mes')
    ax.set_ylabel('% disponible')
    ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'], rotation=30)
    plt.tight_layout()
    plt.show()
    print('Interpretación: Los meses con menor disponibilidad corresponden a temporadas de mayor demanda.')

---
## 11. Posibles transformaciones identificadas

A partir del análisis exploratorio se identificaron los siguientes requerimientos de transformación:

| Colección | Campo | Problema detectado | Acción en transformacion.py |
|---|---|---|---|
| Listings | price | Formato string con $ y , | Normalizar a float |
| Listings | last_scraped, host_since | Tipo string, no datetime | Convertir a YYYY-MM-DD |
| Listings | amenities | Lista serializada como string | Extraer cantidad como variable numérica |
| Listings | minimum_nights | Outliers extremos | Registrar en log, conservar sin eliminar |
| Calendar | available | Valores t/f como string | Convertir a booleano |
| Calendar | date | Tipo string | Convertir a YYYY-MM-DD y derivar mes, año |
| Reviews | comments | Nulos frecuentes | Rellenar con 'Sin comentario' |
| Todos | Duplicados exactos | Registros repetidos | Eliminar con drop_duplicates() |

---
## 12. Resumen de hallazgos del EDA

**Listings:**
- El campo `price` requiere normalización urgente (formato monetario string).
- Variables como `neighbourhood`, `square_feet`, `license` presentan alto porcentaje de nulos y pueden excluirse o imputarse.
- `minimum_nights` tiene outliers extremos que sugieren alquileres de larga temporada.
- El campo `amenities` está serializado como string y debe parsearse para extraer su dimensionalidad.

**Reviews:**
- La columna `comments` tiene nulos que deberán imputarse con un valor neutro.
- La distribución temporal muestra actividad creciente hasta 2019, con caída marcada en 2020 (probable impacto del COVID-19).

**Calendar:**
- El campo `available` usa 't'/'f' en lugar de booleano nativo.
- Se detecta estacionalidad en los precios, con valores más altos en ciertos meses.
- Las transformaciones de fecha permitirán agrupar disponibilidad por semana, mes y trimestre para análisis futuros.